# Clase 10 - Capa GOLD - Agregaciones para analitica

Genera tablas analiticas optimizadas a partir de los datos Silver. Resultado: tabla de hechos agregados lista para BI.

**Prerequisitos:** haber ejecutado `01_Bronze_Ingesta` y `02_Silver_Limpieza_Enriquecimiento`.

Agregaciones que vamos a producir:
1. **Ventas por fecha y tienda** (granularidad diaria por punto de venta).
2. **Ventas por categoria** (agregado de marketing).
3. **Top productos** (ranking simple).

**Nota tecnica:** todas las tablas se persisten como **Managed Tables** de Unity Catalog usando `saveAsTable`. No usamos paths ni `CREATE TABLE ... LOCATION` porque eso genera el error `Missing cloud file system scheme` en este workspace.

## Paso 1 - Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, round as _round, desc

spark = SparkSession.builder.appName("GoldAggClase10").getOrCreate()

spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_diplo_ricardo.gold_clase10")
print("Schema gold_clase10 listo")

## Paso 2 - Leer la tabla Silver

In [ ]:
ventas_silver = spark.read.table("dbx_diplo_ricardo.silver_clase10.silver_ventas")

print("Filas Silver:", ventas_silver.count())
ventas_silver.printSchema()

## Paso 3 - Gold #1: ventas totales por fecha y tienda

In [ ]:
gold_ventas_tienda = (ventas_silver
                      .groupBy("fecha", "nombre_tienda")
                      .agg(
                          _sum("cantidad").alias("total_unidades"),
                          _round(_sum(col("cantidad") * col("precio_unitario")), 2).alias("total_ventas")
                      )
                      .orderBy("fecha", "nombre_tienda")
                     )

gold_ventas_tienda.show(10, truncate=False)

(gold_ventas_tienda.write
                   .format("delta")
                   .mode("overwrite")
                   .option("overwriteSchema", "true")
                   .saveAsTable("dbx_diplo_ricardo.gold_clase10.gold_ventas_por_tienda"))

print("Tabla gold_ventas_por_tienda creada")

## Paso 4 - Gold #2: ventas por categoria

In [ ]:
gold_ventas_categoria = (ventas_silver
                         .groupBy("categoria")
                         .agg(
                             _sum("cantidad").alias("total_unidades"),
                             _round(_sum(col("cantidad") * col("precio_unitario")), 2).alias("total_ventas")
                         )
                         .orderBy(desc("total_ventas"))
                        )

gold_ventas_categoria.show(truncate=False)

(gold_ventas_categoria.write
                      .format("delta")
                      .mode("overwrite")
                      .option("overwriteSchema", "true")
                      .saveAsTable("dbx_diplo_ricardo.gold_clase10.gold_ventas_por_categoria"))

print("Tabla gold_ventas_por_categoria creada")

## Paso 5 - Gold #3: top productos

In [ ]:
gold_top_productos = (ventas_silver
                      .groupBy("producto_id", "nombre_producto", "categoria")
                      .agg(
                          _sum("cantidad").alias("total_unidades"),
                          _round(_sum(col("cantidad") * col("precio_unitario")), 2).alias("total_ventas")
                      )
                      .orderBy(desc("total_ventas"))
                     )

gold_top_productos.show(10, truncate=False)

(gold_top_productos.write
                   .format("delta")
                   .mode("overwrite")
                   .option("overwriteSchema", "true")
                   .saveAsTable("dbx_diplo_ricardo.gold_clase10.gold_top_productos"))

print("Tabla gold_top_productos creada")

## Paso 6 - Verificacion: las 3 tablas Gold deben aparecer

In [ ]:
spark.sql("SHOW TABLES IN dbx_diplo_ricardo.gold_clase10").show(truncate=False)

## Paso 7 - Validacion final con SQL

In [ ]:
%sql
-- Top 10 dias-tienda con mayor venta
SELECT fecha, nombre_tienda, total_unidades, total_ventas
FROM dbx_diplo_ricardo.gold_clase10.gold_ventas_por_tienda
ORDER BY total_ventas DESC
LIMIT 10;

In [ ]:
%sql
SELECT * FROM dbx_diplo_ricardo.gold_clase10.gold_ventas_por_categoria;

In [ ]:
%sql
SELECT * FROM dbx_diplo_ricardo.gold_clase10.gold_top_productos LIMIT 10;